# Train the pneumonia classifier on Kaggle

Before running: attach the chest X-ray dataset and a private Kaggle Dataset containing this project's `src` folder. Enable a GPU in Notebook Settings.

In [ ]:
!pip install -q timm

from pathlib import Path
import shutil

DATASET_INPUT = Path(
    "/kaggle/input/datasets/mirzahamzamustafa/chest-xray-pneumonia/Chest-xray"
)

SOURCE_INPUT = Path(
    "/kaggle/input/datasets/mirzahamzamustafa/pneumonia-project-source"
)

WORKING = Path("/kaggle/working")

assert DATASET_INPUT.exists(), f"X-ray dataset not found: {DATASET_INPUT}"
assert SOURCE_INPUT.exists(), f"Source-code dataset not found: {SOURCE_INPUT}"

# Finds src folder directly, or extracts src.zip if that is what you uploaded.
source_dir = next(SOURCE_INPUT.rglob("src"), None)

if source_dir is None:
    source_archive = next(SOURCE_INPUT.rglob("*.zip"), None)
    assert source_archive is not None, "Could not find src.zip in the source-code dataset."
    shutil.unpack_archive(source_archive, WORKING / "project_source")
    source_dir = next((WORKING / "project_source").rglob("src"), None)

assert source_dir is not None, "Could not find the src folder."

shutil.copytree(source_dir, WORKING / "src", dirs_exist_ok=True)

print("X-ray dataset:", DATASET_INPUT)
print("Source code copied from:", source_dir)

In [ ]:
# Kaggle P100-friendly starting configuration.
!python -m src.train --data-root {DATASET_INPUT} --output-dir /kaggle/working/artifacts --epochs 12 --batch-size 32 --workers 2

In [ ]:
# This evaluates the locked test set once, using the saved validation threshold.
!python -m src.evaluate --data-root {DATASET_INPUT} --checkpoint /kaggle/working/artifacts/best_model.pt --output-dir /kaggle/working/artifacts

Download the entire `/kaggle/working/artifacts` folder from Kaggle Output. Copy it into the local project `artifacts` folder, then run `python app.py` locally.